In [1]:
import os
import json
from pathlib import Path
from collections import defaultdict
import random

In [2]:
base_dir = "/content/drive/MyDrive/Internship/NPTEL"
base_dir = Path(f"{base_dir}/data_v2/")

channels = [
    "delhifoodwalks",
    "main_bhi_bharat",
    "masterchefnambie",
    "northeastindiafood",
    "roohi_haflongbar"
]

In [3]:
category_map = {
    0: [],
    1: [],
    2: []
}

def is_non_empty(value):
    if value is None:
        return False
    if isinstance(value, str):
        return value.strip() != ""
    if isinstance(value, list):
        return len(value) > 0
    return False

In [4]:
for channel in channels:
    folder = base_dir / channel

    for file_path in folder.glob("*.json"):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            description = data.get("metadata", {}).get("description", "")
            transcription_en = data.get("transcription_english", [])

            has_description = is_non_empty(description)
            has_transcription_en = is_non_empty(transcription_en)

            if has_description:
                category_map[0].append(str(file_path))
            elif has_transcription_en:
                category_map[1].append(str(file_path))
            else:
                category_map[2].append(str(file_path))

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

In [5]:
for k in category_map:
    random.shuffle(category_map[k])

In [6]:
num_splits = 4

split_data = [ [] for _ in range(num_splits) ]

for category, files in category_map.items():
    total = len(files)

    # split indices
    chunk_size = total // num_splits
    remainder = total % num_splits

    start = 0

    for i in range(num_splits):
        extra = 1 if i < remainder else 0
        end = start + chunk_size + extra

        chunk = files[start:end]

        # append with label
        split_data[i].extend([(fp, category) for fp in chunk])

        start = end

In [7]:
output_dir = Path("./splits")
output_dir.mkdir(exist_ok=True)

for i, split in enumerate(split_data):
    output_file = output_dir / f"split_{i+1}.txt"

    with open(output_file, "w", encoding="utf-8") as f:
        for file_path, category in split:
            f.write(f"{file_path} {category}\n")

    print(f"Saved: {output_file} | Total: {len(split)}")

Saved: splits/split_1.txt | Total: 587
Saved: splits/split_2.txt | Total: 585
Saved: splits/split_3.txt | Total: 585
Saved: splits/split_4.txt | Total: 584


In [8]:
for i, split in enumerate(split_data):
    counts = defaultdict(int)

    for _, cat in split:
        counts[cat] += 1

    print(f"\nSplit {i+1}")
    print(dict(counts))


Split 1
{0: 461, 1: 111, 2: 15}

Split 2
{0: 460, 1: 111, 2: 14}

Split 3
{0: 460, 1: 111, 2: 14}

Split 4
{0: 460, 1: 110, 2: 14}


In [9]:
from pathlib import Path

splits_dir = Path("splits")

old_prefix = "/content/drive/MyDrive/Internship/NPTEL/data_v2/"
new_prefix = "/content/drive/MyDrive/data_v2/"

for file_path in splits_dir.glob("split_*.txt"):
    updated_lines = []

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # split path and category
        path, category = line.rsplit(" ", 1)

        # replace prefix
        if path.startswith(old_prefix):
            path = path.replace(old_prefix, new_prefix, 1)

        updated_lines.append(f"{path} {category}")

    # overwrite file
    with open(file_path, "w", encoding="utf-8") as f:
        f.write("\n".join(updated_lines) + "\n")

    print(f"Updated: {file_path}")

Updated: splits/split_1.txt
Updated: splits/split_4.txt
Updated: splits/split_3.txt
Updated: splits/split_2.txt


In [10]:
with open("splits/split_1.txt", "r") as f:
    for _ in range(5):
        print(f.readline().strip())

/content/drive/MyDrive/data_v2/delhifoodwalks/572. Tai-Phake Ethnic Food Tour in Namphake.json 0
/content/drive/MyDrive/data_v2/main_bhi_bharat/706. COMING SOON मधय परदश स कहनय.json 0
/content/drive/MyDrive/data_v2/northeastindiafood/106. Yummy Roasted Pork eating with Tomato.json 0
/content/drive/MyDrive/data_v2/main_bhi_bharat/351. टरइबल कचन मथवड क पनय.json 0
/content/drive/MyDrive/data_v2/delhifoodwalks/736. Amritsar Street Food Part 1 shorts.json 0
